# Quantum Drug-Protein Interaction Prediction
## Interactive Demo and Analysis

In [6]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Setup
sys.path.insert(0, str(Path.cwd()))

from src.data.drug_graph import smiles_to_graph, compute_molecular_properties
from src.data.protein_graph import compute_contact_graph
from src.data.pdb_parse import extract_protein_sequence, extract_ligands

print(" Imports successful")

ModuleNotFoundError: No module named 'Bio'

## 1. Drug Molecular Graph Conversion

In [ ]:
# Example SMILES
smiles_examples = {
    'Aspirin': 'CC(=O)Oc1ccccc1C(=O)O',
    'Caffeine': 'CN1C=NC2=C1C(=O)N(C(=O)N2C)C',
    'Ibuprofen': 'CC(C)Cc1ccc(cc1)C(C)C(=O)O',
}

for drug_name, smiles in smiles_examples.items():
    graph = smiles_to_graph(smiles)
    if graph:
        props = compute_molecular_properties(smiles)
        print(f"\n{drug_name}:")
        print(f"  SMILES: {smiles}")
        print(f"  Nodes (atoms): {graph.num_nodes}")
        print(f"  Edges (bonds): {graph.num_edges}")
        print(f"  Properties: {props}")

## 2. Protein Structure Analysis

In [ ]:
# Example PDB file (if available)
pdb_path = Path('data/pdb/1a2b.pdb')

if pdb_path.exists():
    try:
        # Extract sequence
        from src.data.pdb_parse import load_pdb_structure
        structure = load_pdb_structure(str(pdb_path))
        sequence = extract_protein_sequence(structure)
        print(f"PDB: {pdb_path.stem}")
        print(f"Sequence length: {len(sequence) if sequence else 'N/A'}")
        if sequence:
            print(f"Sequence (first 50 AA): {sequence[:50]}...")
        
        # Build contact graph
        graph, aa_seq = compute_contact_graph(str(pdb_path), cutoff=8.0)
        if graph:
            print(f"\nContact Graph:")
            print(f"  Residues: {graph.num_nodes}")
            print(f"  Contact edges: {graph.num_edges}")
        
        # Extract ligands
        ligands = extract_ligands(str(pdb_path))
        if ligands:
            print(f"\nLigands: {list(ligands.keys())}")
    except Exception as e:
        print(f"Note: {e}")
else:
    print(f"PDB file not found at {pdb_path}")
    print("Run: python scripts/generate_sample_data.py")
    print("Or: python scripts/download_pdbs.py --pdb-ids 1a2b")

## 3. Data Loading and Filtering

In [ ]:
from src.data.datasets import load_pdb_ids_and_metadata, load_drugs

# Load metadata
try:
    pdb_metadata, excluded_ids = load_pdb_ids_and_metadata(
        'data/includes/*liganded_*.csv',
        'data/excludes/without_*.txt'
    )
    
    print(f"Loaded {len(pdb_metadata)} proteins")
    print(f"Binding sites: {set(m.get('binding_site', 'unknown') for m in pdb_metadata.values())}")
    
    # Breakdown
    binding_site_counts = {}
    for meta in pdb_metadata.values():
        site = meta.get('binding_site', 'unknown')
        binding_site_counts[site] = binding_site_counts.get(site, 0) + 1
    
    print(f"\nBreakdown by binding site:")
    for site, count in binding_site_counts.items():
        print(f"  {site}: {count}")
except Exception as e:
    print(f"Note: {e}")
    print("Run: python scripts/generate_sample_data.py")

In [ ]:
# Load drugs
try:
    drug_smiles = load_drugs('data/drugs.csv')
    print(f"Loaded {len(drug_smiles)} drugs")
    print(f"\nSample drugs:")
    for drug_id, smiles in list(drug_smiles.items())[:5]:
        print(f"  {drug_id}: {smiles[:50]}...")
except Exception as e:
    print(f"Error: {e}")

## 4. Model Architecture Overview

In [ ]:
import yaml
import torch

# Load config
with open('configs/default.yaml') as f:
    config = yaml.safe_load(f)

print("Model Configuration:")
print(f"  Embedding dim: {config['model']['dim']}")
print(f"  Drug GNN layers: {config['model']['drug_gnn']['layers']}")
print(f"  Protein method: {config['model']['protein']['method']}")
print(f"  Quantum mode: {config['model']['quantum']['mode']}")
print(f"  Quantum qubits: {config['model']['quantum']['n_qubits']}")
print(f"  Quantum depth: {config['model']['quantum']['depth']}")

print(f"\nTraining Configuration:")
print(f"  Batch size: {config['train']['batch_size']}")
print(f"  Learning rate: {config['train']['lr']}")
print(f"  Epochs: {config['train']['epochs']}")
print(f"  Loss: {config['train']['loss']}")

## 5. Dataset Statistics

In [ ]:
try:
    from src.data.datasets import DrugProteinDataset
    
    # Create dataset
    dataset = DrugProteinDataset(
        drug_smiles,
        list(pdb_metadata.keys()),
        pdb_metadata,
        negatives_per_positive=5
    )
    
    # Analyze
    labels = [p[3] for p in dataset.pairs]
    
    print(f"Dataset Statistics:")
    print(f"  Total pairs: {len(dataset)}")
    print(f"  Positives: {sum(labels)}")
    print(f"  Negatives: {len(labels) - sum(labels)}")
    print(f"  Positive ratio: {sum(labels)/len(labels):.2%}")
    
    # Split
    train_pairs, val_pairs, test_pairs = dataset.split()
    print(f"\nData splits:")
    print(f"  Train: {len(train_pairs)}")
    print(f"  Val: {len(val_pairs)}")
    print(f"  Test: {len(test_pairs)}")
except Exception as e:
    print(f"Error: {e}")

## 6. Quick Model Test

In [ ]:
try:
    from src.models.head import DrugProteinInteractionModel
    
    # Build model (small for testing)
    test_config = config.copy()
    test_config['model']['dim'] = 128
    test_config['model']['drug_gnn']['layers'] = 2
    test_config['model']['quantum']['n_qubits'] = 4
    test_config['model']['quantum']['depth'] = 2
    
    model = DrugProteinInteractionModel(test_config)
    
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model created: {n_params:,} parameters")
    print(f" Model is ready for training")
except Exception as e:
    print(f"Warning: {e}")
    print("Check PennyLane installation: pip install pennylane")

## 7. Next Steps

To train the full model:

```bash
# Generate sample data (if needed)
python scripts/generate_sample_data.py

# Train
python run_pipeline.py --config configs/default.yaml --model-type full

# Or use make
make data
make train
```

Results will be saved to `outputs/`